# Intro To Agents Part 1: Tools, MCPs & Tracing
--------------------

1. Introduction

2. Agents & Tools with LangChain

3. Model Context Protocols (MCPs) with FastMCP

4. Agents & Tracing with LangSmith

5. Next Steps

In [1]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
load_dotenv()


True

## 1. Introduction 

In [2]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

model = ChatGroq(model="openai/gpt-oss-120b")
llm  = model | StrOutputParser() 

llm.invoke("Hello")

/Users/mikeharmon/Desktop/mcpweather/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


'Hello! How can I help you today?'

In [3]:
print(llm.invoke("What is the weather like in New York City today?"))

I’m not able to retrieve real‑time weather data, so I can’t tell you the exact conditions in New York City right now. For the most up‑to‑date forecast, you can check a reliable weather service such as:

- **National Weather Service** (weather.gov) – enter “New York, NY” for the official forecast.
- **Weather.com** (The Weather Channel) – provides hourly, daily, and 10‑day outlooks.
- **AccuWeather**, **Weather Underground**, or your favorite weather app on your phone.

In late summer (early August) New York City typically experiences warm to hot temperatures, often in the mid‑80s °F (≈ 29‑30 °C), with occasional thunderstorms or brief rain showers. Humidity can be fairly high, making it feel warmer than the actual temperature.

If you need a quick glance right now, just ask a voice assistant (e.g., Siri, Google Assistant, Alexa) or look at the weather widget on your smartphone. That’ll give you the current temperature, chance of precipitation, and any alerts in real time.


## 2. Agents & Tools with LangChain

In [4]:
from mcps.mcp_server import get_weather

In [5]:
results = get_weather("New York")

In [6]:
results

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 29.46,
  'feels_like': 31.01,
  'temp_min': 26.43,
  'temp_max': 30.08,
  'pressure': 1014,
  'humidity': 55,
  'sea_level': 1014,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 2.24, 'deg': 90, 'gust': 3.58},
 'clouds': {'all': 0},
 'dt': 1786324275,
 'sys': {'type': 2,
  'id': 2111743,
  'country': 'US',
  'sunrise': 1786269626,
  'sunset': 1786320164},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

In [7]:
from langchain.tools import tool


In [8]:
weather_tool = tool("get_weather", get_weather)

In [9]:
type(weather_tool)

langchain_core.tools.structured.StructuredTool

In [10]:
await weather_tool.ainvoke("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 29.46,
  'feels_like': 31.01,
  'temp_min': 26.43,
  'temp_max': 30.08,
  'pressure': 1014,
  'humidity': 55,
  'sea_level': 1014,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 2.24, 'deg': 90, 'gust': 3.58},
 'clouds': {'all': 0},
 'dt': 1786324275,
 'sys': {'type': 2,
  'id': 2111743,
  'country': 'US',
  'sunrise': 1786269626,
  'sunset': 1786320164},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

In [11]:
agents = create_agent(model=model, tools=[weather_tool])

In [12]:
query = "What is the weather in New York?"

In [13]:
result = await agents.ainvoke({'messages': [{'role': 'user', 'content': query}]})

In [14]:
messages = result.get("messages")

In [15]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='284a7855-ad87-42d0-9dba-b10546d9e49f'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in New York. We need to fetch via function get_weather. Use city "New York". Probably need to call function.', 'tool_calls': [{'id': 'fc_ac456665-7e22-4029-8402-ac051b613830', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 261, 'total_tokens': 319, 'completion_time': 0.120159789, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.012392281, 'prompt_tokens_details': None, 'queue_time': 0.106895422, 'total_time': 0.13255207}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--

## 3. Model Context Protocols (MCPs) with FastMCP

In [16]:
from fastmcp import Client

In [17]:
mcp_client = Client("http://localhost:8000/mcp")

In [18]:
async with mcp_client:
    tools = await mcp_client.list_tools()

In [53]:
tools

[Tool(name='get_weather', title=None, description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', inputSchema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, outputSchema={'additionalProperties': True, 'type': 'object'}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='convert_address_to_point', title=None, description='Convert a street address to a Shapely Point object.', inputSchema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'description': 'A free‑form US address (e

In [54]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York"})

In [55]:
print(result)

CallToolResult(content=[TextContent(type='text', text='{"coord":{"lon":-74.006,"lat":40.7143},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01n"}],"base":"stations","main":{"temp":29.53,"feels_like":31.12,"temp_min":26.65,"temp_max":30.08,"pressure":1014,"humidity":55,"sea_level":1014,"grnd_level":1014},"visibility":10000,"wind":{"speed":2.24,"deg":334,"gust":2.68},"clouds":{"all":0},"dt":1786323757,"sys":{"type":2,"id":2111743,"country":"US","sunrise":1786269626,"sunset":1786320164},"timezone":-14400,"id":5128581,"name":"New York","cod":200}', annotations=None, meta=None)], structured_content={'coord': {'lon': -74.006, 'lat': 40.7143}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 29.53, 'feels_like': 31.12, 'temp_min': 26.65, 'temp_max': 30.08, 'pressure': 1014, 'humidity': 55, 'sea_level': 1014, 'grnd_level': 1014}, 'visibility': 10000, 'wind': {'speed': 2.24, 'deg': 334, 'gust': 2.6

In [19]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "http://localhost:8000/mcp", "transport": "http"}})

In [31]:
tool_list = await lc_client.get_tools()

In [32]:
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x131bc4a40>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

In [33]:
weather_mcp_tool = [tool_list[0]]

In [34]:
weather_mcp_tool

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x131bc4a40>)]

In [35]:
weather_agent = create_agent(model=model, tools=weather_mcp_tool)

In [36]:
result = await weather_agent.ainvoke({'messages': [{
    'role': 'user', 'content': 'What is the weather in New York?'}
    ]})

In [37]:
result.get("messages")

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='72be0354-2556-44ab-8b3a-890e248c36c6'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to fetch weather for New York using function. Use get_weather with city "New York".', 'tool_calls': [{'id': 'fc_7b777a56-d70c-4daa-8d0c-c6d928e74a54', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 196, 'total_tokens': 245, 'completion_time': 0.102847803, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.008390368, 'prompt_tokens_details': None, 'queue_time': 0.083510456, 'total_time': 0.111238171}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe93e-78a4-7002-a79b-910e4ecf455b-0

## 4. Agents & Traces With LangSmith


Others tools in MCP Server

In [38]:

from shapely.geometry import Point

In [39]:
async with mcp_client:
    result = await mcp_client.call_tool("convert_address_to_point", {"address": "98 Thomas St, New York, NY 10013"})

In [40]:
result.content

[TextContent(type='text', text='{"lat":40.7172707,"lng":-74.0087569}', annotations=None, meta=None)]

In [42]:
agent = create_agent(model=model, tools=tool_list)

In [43]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct 306 West 54th Street, manhattan, manhattan in?'
}]})

ToolException: Error calling tool 'get_precinct_info': [Errno 2] No such file or directory: 'notebooks/precinct_info.parquet'

In [ ]:
print(result.get("messages")[-1].content)

AttributeError: 'CallToolResult' object has no attribute 'get'

In [26]:
result = await mcp_agent.ainvoke({
    'messages': [{'role': 'user', 'content': 'What temperature at 13 Jefferson Road, Brookline MA. 02467?'
}]})

NameError: name 'mcp_agent' is not defined

In [25]:
print(result.get("messages")[-1].content)

AttributeError: 'CallToolResult' object has no attribute 'get'

In [140]:
precinct_df['precinct_number'] = precinct_df['Precinct'].apply(get_precinct_number)

In [142]:
precinct_df.to_parquet("../notebooks/precinct_info.parquet")

In [ ]:
def get_precinct_info(precinct_number: int) -> dict:
    """
    Return the precinct information for a given precinct number.

    Parameters
    ----------
    precinct_number: int
        The precinct number (e.g. 1, 5, 14, etc.).

    Returns
    -------
    dict
        A dictionary containing the precinct information (e.g. name, address, etc.).
        Returns an empty dictionary if the precinct number is not found.
    """
    df = pd.read_parquet("../notebooks/precinct_info.parquet")
    precinct_info = df[df["precinct_number"] == precinct_number].to_dict(orient="records")
    return precinct_info[0] if precinct_info else {}

In [144]:
precinct_info_tool = tool("get_precinct_info", get_precinct_info)

In [145]:
tool_list.append(precinct_info_tool)

In [146]:
my_agent = create_agent(model=llm, tools=tool_list)

In [147]:
result = await my_agent.ainvoke({
    'messages': [{
        'role': 'user', 
        'content': 'What the phone number for the precinct that covers 567 Ocean Avenue, Brooklyn, NY?'
}]})

In [148]:
print(result.get("messages")[-1].content)

The address 567 Ocean Avenue, Brooklyn, NY falls within the **70th Precinct**.  
**Phone:** **718‑851‑5511**.


In [149]:
result

{'messages': [HumanMessage(content='What the phone number for the precinct that covers 567 Ocean Avenue, Brooklyn, NY?', additional_kwargs={}, response_metadata={}, id='63a18c05-7858-4317-b4b2-ac5a9a1db32b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need phone number for precinct covering address. Use get_police_precinct then get_precinct_info.', 'tool_calls': [{'id': 'fc_ac1025b3-84fc-4788-ba12-232f7a052754', 'function': {'arguments': '{"address":"567 Ocean Avenue, Brooklyn, NY"}', 'name': 'get_police_precinct'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 501, 'total_tokens': 559, 'completion_time': 0.121600815, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.022343348, 'prompt_tokens_details': None, 'queue_time': 0.128569714, 'total_time': 0.143944163}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c56a34c3f3', 'service_tier': 'on_demand', 'finish_reason': 'too

## bathrooms 

In [245]:
def find_closest_restroom(lat_longs: Dict[str, float]) -> dict:
    """
    Find the closest public restroom to a given point.

    Parameters
    ----------
    lat_longs: Dict[str, float]
        A dictionary containing the latitude and longitude of the location to search from.

    Returns
    -------
    dict
        A dictionary containing the information of the closest restroom.
        Returns an empty dictionary if no restrooms are found.
    """
    gdf = gpd.read_file("../Public_Restrooms_20260508.geojson")
    point = Point(lat_longs['lng'], lat_longs['lat'])
    gdf["dist_m"] = gdf.distance(point)
    closest_row = gdf.loc[gdf["dist_m"].idxmin()]
    return closest_row.to_dict() if not closest_row.empty else {}

In [246]:
closest_restroom_tool = tool("find_closest_restroom", find_closest_restroom)

In [247]:
tools = [weather_tool, address_to_point_tool, precinct_tool, closest_restroom_tool]

In [248]:
final_agent = create_agent(model=llm, tools=tools)

In [255]:
result = await final_agent.ainvoke({
    "messages": [{
        'role': 'user', 
        'content': 'Whats closets bathroom in 567 Ocean Ave, Brooklyn, NY?'}
        ]})

print(result.get("messages")[-1].content)

/var/folders/zr/sxbhzdlx79gfs1wc_mlk0f7h0000gn/T/ipykernel_68599/958974888.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["dist_m"] = gdf.distance(point)


The nearest public restroom to **567 Ocean Ave, Brooklyn, NY** is located at the **Flatbush Library (BPL)**.

**Details**

| Item | Information |
|------|--------------|
| **Facility** | Flatbush Library (Brooklyn Public Library) |
| **Address** | 9 Willoughby St, Brooklyn, NY 11213 (the library’s main entrance) |
| **Coordinates** | 40.651933 ° N, ‑73.958244 ° W |
| **Distance from your location** | ~0.0037 km (≈ 4 m) – essentially right next door |
| **Restroom type** | Multi‑stall women/men restrooms |
| **Accessibility** | Partially accessible (some stalls are wheelchair‑friendly) |
| **Changing stations** | Yes (family‑friendly changing stations) |
| **Hours of operation** | <pre>Monday  10 am – 6 pm<br>Tuesday  1 pm – 8 pm<br>Wednesday 10 am – 6 pm<br>Thursday 10 am – 8 pm<br>Friday  10 am – 6 pm<br>Saturday 10 am – 5 pm<br>Sunday  CLOSED</pre> |
| **Website** | <https://www.bklynlibrary.org/locations/flatbush> |

**How to get there**

1. Exit the building at 567 Ocean Ave.  
2. 

**Closest public restroom**

| Item | Details |
|------|---------|
| **Facility** | **Flatbush Library (Brooklyn Public Library)** |
| **Address** | 2273 Flatbush Ave, Brooklyn, NY 11226 |
| **Type** | Library‑operated, multi‑stall restrooms (women’s & men’s) with changing stations |
| **Accessibility** | Partially accessible (some stalls equipped for wheelchair users) |
| **Hours (2024‑2025)** | • Monday 10 am – 6 pm  <br>• Tuesday 1 pm – 8 pm  <br>• Wednesday 10 am – 6 pm  <br>• Thursday 10 am – 8 pm  <br>• Friday 10 am – 6 pm  <br>• Saturday 10 am – 5 pm  <br>• Sunday CLOSED |
| **Status** | Operational (Year‑round) |
| **Website** | <https://www.bklynlibrary.org/locations/flatbush> |
| **Distance from 567 Ocean Ave** | ≈ 0.38 km (≈ 0.24 mi or 400 ft) – about a 5‑minute walk |
| **Walking directions** | 1. Head **southwest** on Ocean Ave toward Flatbush Ave. <br>2. Turn **right** onto Flatbush Ave (about 0.2 mi). <br>3. The library is on the **right‑hand side**, at 2273 Flatbush Ave